In [1]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../data/raw")

sku_master = pd.read_csv(DATA_PATH / "sku_master.csv")
calendar = pd.read_csv(DATA_PATH / "calendar.csv")
sales_daily = pd.read_csv(DATA_PATH / "sales_daily.csv")
inventory_snapshots = pd.read_csv(
    DATA_PATH / "inventory_snapshots.csv"
)

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [2]:
# Convert date columns

calendar["date"] = pd.to_datetime(calendar["date"])
sales_daily["date"] = pd.to_datetime(sales_daily["date"])
inventory_snapshots["date"] = pd.to_datetime(
    inventory_snapshots["date"]
)
sku_master["launch_date"] = pd.to_datetime(
    sku_master["launch_date"]
)

print("Date conversion completed.")

Date conversion completed.


In [3]:
# Handle missing promotion values

calendar["promotion_event"] = (
    calendar["promotion_event"].fillna("None")
)

print("Missing values after cleaning:")
print(calendar.isnull().sum())

Missing values after cleaning:
date               0
year               0
month              0
week               0
day_of_week        0
quarter            0
is_weekend         0
season             0
holiday_flag       0
promotion_event    0
dtype: int64


In [4]:
# Remove duplicate rows

sku_master = sku_master.drop_duplicates()
calendar = calendar.drop_duplicates()
sales_daily = sales_daily.drop_duplicates()
inventory_snapshots = inventory_snapshots.drop_duplicates()

print("Duplicate rows removed.")

Duplicate rows removed.


In [5]:
# Convert important numeric columns

sales_daily["units_sold"] = pd.to_numeric(
    sales_daily["units_sold"], errors="coerce"
)

sales_daily["revenue"] = pd.to_numeric(
    sales_daily["revenue"], errors="coerce"
)

sales_daily["unit_price"] = pd.to_numeric(
    sales_daily["unit_price"], errors="coerce"
)

inventory_snapshots["on_hand_units"] = pd.to_numeric(
    inventory_snapshots["on_hand_units"], errors="coerce"
)

inventory_snapshots["on_order_units"] = pd.to_numeric(
    inventory_snapshots["on_order_units"], errors="coerce"
)

inventory_snapshots["inventory_value"] = pd.to_numeric(
    inventory_snapshots["inventory_value"], errors="coerce"
)

print("Numeric columns converted.")

Numeric columns converted.


In [6]:
# Merge sales with calendar information

analysis_ready = sales_daily.merge(
    calendar,
    on="date",
    how="left"
)

# Merge SKU information

analysis_ready = analysis_ready.merge(
    sku_master,
    on="sku",
    how="left"
)

print("Analysis-ready dataset created.")
print("Shape:", analysis_ready.shape)

analysis_ready.head()

Analysis-ready dataset created.
Shape: (365200, 21)


,date,sku,units_sold,revenue,unit_price,promotion_flag,year,month,week,day_of_week,...,is_weekend,season,holiday_flag,promotion_event,category,subcategory,launch_date,unit_cost,list_price,lead_time_days
0,2021-01-01,SKU0001,0,0.0,4483.83,0,2021,1,53,4,...,0,Winter,0,None,Furniture,Chair,2021-03-22,2734.88,4483.83,26
1,2021-01-02,SKU0001,0,0.0,4483.83,0,2021,1,53,5,...,1,Winter,0,None,Furniture,Chair,2021-03-22,2734.88,4483.83,26
2,2021-01-03,SKU0001,0,0.0,4483.83,0,2021,1,53,6,...,1,Winter,0,None,Furniture,Chair,2021-03-22,2734.88,4483.83,26
3,2021-01-04,SKU0001,0,0.0,4483.83,0,2021,1,1,0,...,0,Winter,0,None,Furniture,Chair,2021-03-22,2734.88,4483.83,26
4,2021-01-05,SKU0001,0,0.0,4483.83,0,2021,1,1,1,...,0,Winter,0,None,Furniture,Chair,2021-03-22,2734.88,4483.83,26


In [7]:
# Create processed data folder

PROCESSED_PATH = Path("../data/processed")
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print("Processed folder ready.")

Processed folder ready.


In [8]:
# Save analysis-ready dataset

output_file = PROCESSED_PATH / "analysis_ready.csv"

analysis_ready.to_csv(
    output_file,
    index=False
)

print("Saved successfully:")
print(output_file)

Saved successfully:
..\data\processed\analysis_ready.csv


In [9]:
print("Final dataset shape:", analysis_ready.shape)

print("\nMissing values:")
print(analysis_ready.isnull().sum())

print("\nDuplicate rows:",
      analysis_ready.duplicated().sum())

Final dataset shape: (365200, 21)

Missing values:
date               0
sku                0
units_sold         0
revenue            0
unit_price         0
promotion_flag     0
year               0
month              0
week               0
day_of_week        0
quarter            0
is_weekend         0
season             0
holiday_flag       0
promotion_event    0
category           0
subcategory        0
launch_date        0
unit_cost          0
list_price         0
lead_time_days     0
dtype: int64

Duplicate rows: 0
